# 🎯 Multi-Camera Person Re-Identification System
## Phase 1 — Offline Video Processing (GPU Accelerated)

This notebook runs the full ReID pipeline on Google Colab with GPU.

**Pipeline:** YOLOv8 → ByteTrack → OSNet → Dual-Embedding Gallery → ST-Gating

**Identity Stability Features (v3):**
- ✅ Dynamic Embedding Bank (Feature Clustering for different angles/lighting)
- ✅ Cooldown buffer for ByteTrack track reassignments
- ✅ Adaptive thresholds (same-camera vs cross-camera)
- ✅ Reduced EMA drift with update cap
- ✅ Relaxed ST-gating for concurrent camera views

**Steps:**
1. Install dependencies
2. Upload your CCTV clips
3. Run the pipeline
4. View & download annotated videos

---

## 1. Check GPU

In [4]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Fri Jun  5 09:39:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [5]:
# Core dependencies
!pip install -q ultralytics faiss-cpu scipy opencv-python pyyaml lap

# torchreid (ReID backbone — needs build deps first)
!pip install -q Cython gdown tensorboard h5py
!pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git

print("\n✅ All dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 11.0 MB/s eta 0:00:00

✅ All dependencies installed!


In [6]:
# Verify all imports
import torchreid; print(f"✅ torchreid {torchreid.__version__}")
import torch; print(f"✅ torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
from ultralytics import YOLO; print(f"✅ ultralytics")
import faiss; print(f"✅ faiss")
import scipy; print(f"✅ scipy {scipy.__version__}")
import cv2; print(f"✅ opencv {cv2.__version__}")

✅ torchreid 1.4.0
✅ torch 2.10.0+cu128 | CUDA: True
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ ultralytics
✅ faiss
✅ scipy 1.16.3
✅ opencv 4.13.0


## 3. Clone / Upload Project Code

**Option A:** Clone from git (if you have a repo)  
**Option B:** Upload the `multi_camera_reid/` folder manually  
**Option C:** Mount Google Drive and copy from there

In [ ]:
# Option A: Clone from git
!git clone https://github.com/hypertonny/person-reid.git /content/person-reid

# Option B: Upload files manually
# Use the file browser on the left sidebar to upload multi_camera_reid/ folder
# Or mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

# Then copy from Drive (adjust path):
# !cp -r /content/drive/MyDrive/person-reid/multi_camera_reid /content/multi_camera_reid

In [7]:
# Set working directory (adjust if you used a different path)
import os

# If cloned from git:
PROJECT_DIR = "/content/person-reid/multi_camera_reid"

# If uploaded manually or from Drive:
# PROJECT_DIR = "/content/multi_camera_reid"

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

Working directory: /content/multi_camera_reid
Files: ['data']


In [8]:
# Create required directories
import os
for d in ['data/clips', 'models', 'outputs/annotated', 'outputs/gallery_state', 'logs']:
    os.makedirs(d, exist_ok=True)
    print(f"  ✅ {d}/")

  ✅ data/clips/
  ✅ models/
  ✅ outputs/annotated/
  ✅ outputs/gallery_state/
  ✅ logs/


## 4. Upload CCTV Clips

Upload your two video clips. Choose ONE method below:

In [10]:
# Method 1: Upload from your computer (interactive upload)
from google.colab import files
print("Upload cctv1.mp4 and cctv2.mp4:")
uploaded = files.upload()

import shutil
for filename in uploaded.keys():
    if 'cctv1' in filename.lower() or '1' in filename:
        shutil.move(filename, 'data/clips/cctv1.mp4')
        print(f"  → {filename} → data/clips/cctv1.mp4")
    elif 'cctv2' in filename.lower() or '2' in filename:
        shutil.move(filename, 'data/clips/cctv2.mp4')
        print(f"  → {filename} → data/clips/cctv2.mp4")

Upload cctv1.mp4 and cctv2.mp4:


KeyboardInterrupt: 

In [11]:
# Method 2: Copy from Google Drive (uncomment and adjust paths)
# !cp "/content/drive/MyDrive/cctv1.mp4" data/clips/cctv1.mp4
# !cp "/content/drive/MyDrive/cctv2.mp4" data/clips/cctv2.mp4

# Verify clips are in place
!ls -lh data/clips/

total 5.1M
-rw------- 1 root root 3.0M Jun  5 09:39 cctv1.mp4
-rw------- 1 root root 2.1M Jun  5 09:39 cctv2.mp4


## 5. Configure

The default `config.yaml` includes all v2 identity stability fixes.  
Key parameters you may want to tune:

In [16]:
# View current config
!cat config.yaml

# ============================================================
# Multi-Camera Person ReID — Configuration File
# Phase 1: Offline video clip processing
# Phase 2: Switch sources to rtsp:// URLs and set mode: live
# ============================================================

# --- Runtime Mode ---
runtime:
  mode: "offline"          # "offline" (Phase 1) | "live" (Phase 2)
  frame_skip: 1            # Process every Nth frame (1 = every frame, 2 = every other, etc.)
  display: false           # Show live cv2 windows (set true for Phase 2 or debug)
  gallery_autosave_sec: 300  # Auto-save gallery every N seconds (Phase 2)

# --- Camera Sources ---
# Phase 1: Use file paths under data/clips/
# Phase 2: Replace source with rtsp://user:pass@ip:port/stream
cameras:
  - id: "CCTV-1"
    source: "data/clips/cctv1.mp4"
    start_offset_sec: 0.0    # When does this clip start relative to shared timeline?

  - id: "CCTV-2"
    source: "data/clips/cctv2.mp4"
    start_offset_sec: 0.0    # Adjust 

In [17]:
# Optional: Override specific config values for Colab GPU
import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Force CUDA device for GPU acceleration
config['reid']['device'] = 'cuda'

# --- Identity Stability Tuning (v2 defaults) ---
# Cross-camera cosine threshold (higher = stricter matching)
# config['matching']['cosine_threshold'] = 0.70

# Same-camera occlusion recovery threshold (lower = more forgiving)
# config['matching']['same_camera_threshold'] = 0.55

# EMA update rate (lower = more stable prototypes, less drift)
# config['matching']['ema_alpha'] = 0.03

# Cooldown buffer duration (seconds to keep disappeared tracks)
# config['matching']['track_cooldown_sec'] = 10.0

# Spatio-temporal travel time (tune to your floor plan)
# config['spatio_temporal']['camera_pairs'][0]['mu_sec'] = 15.0
# config['spatio_temporal']['camera_pairs'][0]['sigma_sec'] = 30.0

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print("✅ Config updated — CUDA enabled")

✅ Config updated — CUDA enabled


## 6. 🚀 Run the Pipeline!

In [18]:
# Run the full multi-camera ReID pipeline
!python orchestrator.py --config config.yaml

python3: can't open file '/content/multi_camera_reid/orchestrator.py': [Errno 2] No such file or directory


## 7. View Results

In [19]:
# Check output files
!ls -lh outputs/annotated/
print("\n--- Runtime Log (last 50 lines) ---")
!tail -50 logs/reid_runtime.log

total 0

--- Runtime Log (last 50 lines) ---
tail: cannot open 'logs/reid_runtime.log' for reading: No such file or directory


In [ ]:
# Quick identity stability check — count unique IDs per camera
!echo "=== New ID creation events ==="
!grep -c '✨ New person' logs/reid_runtime.log || echo '0'
!echo ""
!echo "=== Cooldown recoveries (prevented ID switches) ==="
!grep -c 'Cooldown recovery' logs/reid_runtime.log || echo '0'
!echo ""
!echo "=== SESSION SUMMARY ==="
!grep -A3 'SESSION SUMMARY' logs/reid_runtime.log | tail -4

In [ ]:
# Preview annotated video inline (first N frames as images)
import cv2
from IPython.display import display, Image
import numpy as np

def show_video_frames(video_path, n_frames=5, interval=None):
    """Display N evenly-spaced frames from a video."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if interval is None:
        interval = max(1, total // n_frames)

    print(f"\n📹 {video_path} — {total} frames")
    for i in range(0, min(total, n_frames * interval), interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            # Resize for display
            h, w = frame.shape[:2]
            scale = 640 / w
            frame = cv2.resize(frame, (640, int(h * scale)))
            _, buf = cv2.imencode('.jpg', frame)
            display(Image(data=buf.tobytes(), width=640))
            print(f"  Frame {i}/{total}")
    cap.release()

show_video_frames('outputs/annotated/cctv-1_annotated.mp4', n_frames=4)
show_video_frames('outputs/annotated/cctv-2_annotated.mp4', n_frames=4)

In [ ]:
# Side-by-side comparison using HTML5 video (if running Colab in browser)
from IPython.display import HTML
import subprocess

# Convert to H.264 for browser playback
for cam in ['cctv-1', 'cctv-2']:
    src = f'outputs/annotated/{cam}_annotated.mp4'
    dst = f'outputs/annotated/{cam}_web.mp4'
    subprocess.run(
        ['ffmpeg', '-y', '-i', src, '-vcodec', 'libx264', '-acodec', 'aac', dst],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print(f"✅ Converted {cam}")

print("\n🎬 Videos ready for download!")
!ls -lh outputs/annotated/*_web.mp4

## 8. Download Results

In [ ]:
# Download annotated videos to your computer
from google.colab import files

for f in ['outputs/annotated/cctv-1_web.mp4', 'outputs/annotated/cctv-2_web.mp4']:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)

# Also download the gallery state
if os.path.exists('outputs/gallery_state/gallery_final.pkl'):
    files.download('outputs/gallery_state/gallery_final.pkl')

In [ ]:
# Or copy to Google Drive
# !mkdir -p /content/drive/MyDrive/reid_results/
# !cp outputs/annotated/*_web.mp4 /content/drive/MyDrive/reid_results/
# !cp outputs/gallery_state/gallery_final.pkl /content/drive/MyDrive/reid_results/
# print("✅ Results saved to Google Drive")